# 第10章 自监督学习
## Self-Supervised Learning — BERT与GPT

**来源：李宏毅《深度学习教程》第10章 | 对应原书第177-201页**

---

## 一、知识地图：全章结构与脉络

```
第10章 自监督学习 (SSL)
├── 10.1 BERT: 来自Transformers的双向编码器表示
│   ├── BERT架构：Transformer编码器
│   ├── 预训练任务1：掩码语言模型 (MLM)
│   │   ├── 随机掩码15%的token
│   │   ├── 80%替换为[MASK]
│   │   ├── 10%替换为随机词
│   │   └── 10%保持原词
│   ├── 预训练任务2：下一句预测 (NSP)
│   │   ├── [CLS] + 句A + [SEP] + 句B + [SEP]
│   │   └── 预测B是不是A的下一句
│   ├── BERT的使用方式（微调）
│   │   ├── 情况1：情感分析（单句分类）
│   │   ├── 情况2：词性标注（逐token分类）
│   │   ├── 情况3：自然语言推理（两句关系）
│   │   └── 情况4：抽取式问答（预测起始/结束位置）
│   ├── 10.1.2 BERT为什么有效？
│   │   ├── 上下文相关词嵌入
│   │   ├── “吃苹果” vs "苹果手机"中的"果"
│   │   └── CBOW词袋模型的深度学习版
│   └── 10.1.3 BERT的变种
│       ├── 多语言BERT（104种语言）
│       ├── 跨语言零样本迁移
│       ├── 中英文嵌入对齐
│       └── 无监督token级翻译
├── 10.2 GPT: 生成式预训练
│   ├── 预训练任务：预测下一个token
│   ├── 单向（因果）注意力掩码
│   ├── In-Context Learning (语境学习)
│   │   ├── 零样本学习 (Zero-shot)
│   │   ├── 单样本学习 (One-shot)
│   │   └── 小样本学习 (Few-shot)
│   └── GPT-3：1750亿参数
└── 自监督学习在其他领域
    ├── 计算机视觉：SimCLR、BYOL
    └── 语音：SUPERB基准
```

## 二、什么是自监督学习？

### 2.1 自监督学习 = 无标注数据的监督学习

Yann LeCun在2019年提出了"自监督学习"（SSL）一词。核心思想：

**监督学习**：需要人工标注的标签（贵、慢、少）

**自监督学习**：从数据本身自动生成标签，不需要人工标注

具体做法：
1. 取一段无标注数据 $x$
2. 将 $x$ 分成两部分：$x'$（模型输入）和 $x''$（预测目标/伪标签）
3. 让模型根据 $x'$ 预测 $x''$
4. 用预测误差训练模型

> **类比**：就像学生自己看书出题考自己——把书中的一段遮住，尝试回忆被遮的内容。不需要老师出题，书本自己就是老师。

### 2.2 为什么叫"自监督"而不叫"无监督"？

无监督学习是一个大类（聚类、降维等），自监督学习只是其中**有明确伪标签**的那一种。为精准起见，单独命名为"自监督学习"。

### 2.3 模型的规模

| 模型 | 参数量 |
|------|--------|
| ELMo | 94M |
| BERT | 340M |
| GPT-2 | 1.5B |
| GPT-3 | 175B |
| Switch Transformer | 1.6T |

模型越大，能力越强——这就是"规模法则（Scaling Law）"。

## 三、BERT：填空游戏训练出的理解能力

### 3.1 BERT的架构

BERT = **Transformer的编码器**。其架构与Transformer编码器完全相同：多层自注意力+残差连接+层归一化+前馈网络。

输入一行向量，输出另一行向量，长度与输入相同。

### 3.2 预训练任务一：掩码语言模型 (MLM)

随机掩码15%的输入token，让BERT预测被掩码的内容。

**三种掩码策略（随机选择）**：
- 80%：替换为特殊token `[MASK]`
- 10%：替换为随机词（如"深度"→"一天"）
- 10%：保持原词不变（但仍然要预测）

**为什么需要后两种策略？** 如果只用[MASK]，BERT会认为训练后只会遇到[MASK]标签，但微调阶段下游任务可能没有[MASK]——造成训练-测试不匹配。加入随机替换和保持不变能让BERT学会在非[MASK]位置也能输出正确的词。

**训练方法**：在BERT的输出上接一个线性变换+softmax，做分类问题（如果词表有4000个汉字，就是4000类的分类问题）。使用交叉熵损失训练，联合训练BERT和线性层。

### 3.3 预训练任务二：下一句预测 (NSP)

输入：`[CLS] 句A [SEP] 句B [SEP]`

取`[CLS]`对应的输出，通过线性层做二分类：句B是不是句A的下一句？

**但后来发现NSP用处不大**。RoBERTa论文明确指出NSP几乎没用。原因：这个任务太简单——随机从数据库取两句，它们大概率完全不相关，BERT轻易就能分辨。

**改进版：句序预测 (SOP)**，被ALBERT采用。给两个相连的句子，让BERT判断哪个在前哪个在后——难度更大，更有用。

## 四、BERT的微调：四种下游任务

预训练完成后，BERT只是一个"会填空的模型"，不能直接用于实际任务。需要通过**微调（Fine-tuning）** 适配到具体下游任务。

### 4.1 情况1：情感分析（输入序列，输出类别）

输入：`[CLS] 这个电影很好看 [SEP]`

取`[CLS]`对应的输出 → 线性变换 → softmax → 正面/负面

训练时：线性层随机初始化，BERT部分用预训练权重初始化。两者一起用梯度下降更新。

### 4.2 情况2：词性标注（输入输出等长序列）

输入：`[CLS] 我 爱 深度 学习`

每个token的输出 → 各自的线性变换 → 各自的词性预测

### 4.3 情况3：自然语言推理 (NLI)

输入两个句子，判断关系（蕴含/矛盾/中立）。

输入：`[CLS] 前提 [SEP] 假设 [SEP]`

取`[CLS]`输出 → 分类（蕴含/矛盾/中立）

### 4.4 情况4：抽取式问答

给文章和问题，答案必须是文章中的一个连续片段。

输入：`[CLS] 问题 [SEP] 文章 [SEP]`

训练两个向量（橙色=起始位置，蓝色=结束位置）：
1. 橙色向量与文章各token的输出做内积 → softmax → 起始位置 $s$
2. 蓝色向量与文章各token的输出做内积 → softmax → 结束位置 $e$
3. 答案 = 文章的第$s$到第$e$个token

只有橙色和蓝色向量是随机初始化的，BERT部分用预训练权重。

In [ ]:
# ============================================
# PyTorch示例1：BERT微调框架（4种下游任务）
# ============================================
import torch
import torch.nn as nn

class BertForDownstreamTasks:
    """BERT微调框架——展示四种下游任务的设计模式"""
    pass

# 情况1：情感分析
class BertForSentimentAnalysis(nn.Module):
    def __init__(self, bert_encoder, hidden_size=768, num_classes=2):
        super().__init__()
        self.bert = bert_encoder  # 预训练的Transformer编码器
        self.classifier = nn.Linear(hidden_size, num_classes)
    
    def forward(self, input_ids):
        outputs = self.bert(input_ids)  # [B, seq_len, hidden]
        cls_output = outputs[:, 0, :]    # 取[CLS]的输出
        return self.classifier(cls_output)

# 情况2：词性标注
class BertForPOSTagging(nn.Module):
    def __init__(self, bert_encoder, hidden_size=768, num_tags=50):
        super().__init__()
        self.bert = bert_encoder
        self.classifier = nn.Linear(hidden_size, num_tags)
    
    def forward(self, input_ids):
        outputs = self.bert(input_ids)  # [B, seq_len, hidden]
        return self.classifier(outputs)  # 每个token都分类

# 情况3：自然语言推理
class BertForNLI(nn.Module):
    def __init__(self, bert_encoder, hidden_size=768, num_classes=3):
        super().__init__()
        self.bert = bert_encoder
        self.classifier = nn.Linear(hidden_size, num_classes)
    
    def forward(self, input_ids):
        # 输入：[CLS] 前提 [SEP] 假设 [SEP]
        outputs = self.bert(input_ids)
        cls_output = outputs[:, 0, :]  # 取[CLS]
        return self.classifier(cls_output)  # 蕴含/矛盾/中立

# 情况4：抽取式问答
class BertForQA(nn.Module):
    def __init__(self, bert_encoder, hidden_size=768):
        super().__init__()
        self.bert = bert_encoder
        # 起始位置和结束位置各一个可训练向量
        self.start_vector = nn.Parameter(torch.randn(hidden_size))
        self.end_vector = nn.Parameter(torch.randn(hidden_size))
    
    def forward(self, input_ids):
        outputs = self.bert(input_ids)  # [B, seq_len, hidden]
        # 计算每个位置是起始/结束的分数
        start_scores = torch.matmul(outputs, self.start_vector)  # [B, seq_len]
        end_scores = torch.matmul(outputs, self.end_vector)
        return start_scores, end_scores

print("BERT四种下游任务的设计模式已展示。")
print()
print("共同特点：")
print("- BERT部分使用预训练权重初始化")
print("- 新增的线性层/向量随机初始化")
print("- 两者一起用梯度下降微调")

## 五、BERT为什么有效？——上下文相关词嵌入

### 5.1 核心能力：同一个词，不同上下文 → 不同向量

"吃苹果"中的"果" vs "苹果手机"中的"果" → BERT给它们不同的嵌入向量！

实验验证：收集10个含"果"的句子（5个=食物苹果，5个=苹果公司），计算两两余弦相似度。
- 食物苹果之间：高相似度
- 苹果公司之间：高相似度
- 食物苹果 vs 苹果公司：**低相似度**

> BERT真的理解了词义的差异！

### 5.2 语言学理论的支持

语言学家John Rupert Firth在1960年代提出：**"You shall know a word by the company it keeps."**（要理解一个词，看它的"朋友圈"——经常一起出现的词。）

BERT在填空过程中，必须从上下文中提取信息来预测被遮的词。所以它输出的向量就是上下文信息的精华——这就是"上下文相关词嵌入"（Contextualized Word Embedding）。

### 5.3 BERT是"深度版CBOW"

CBOW（连续词袋模型）：也是把中间挖空，用上下文预测中间词——和BERT的MLM一模一样！只是CBOW用浅层线性模型（当时算力不够），BERT用深层Transformer。

### 5.4 令人惊讶的发现：BERT不仅理解语义

把DNA序列（ATCG）映射为无意义的"单词"输入BERT，BERT仍然能对DNA进行分类！

这说明BERT的成功不完全来自"理解语义"——可能它本质上提供了一组**特别好用的初始化参数**，适合训练大规模模型。这个问题至今没有完全解答。

## 六、BERT的变种：多语言BERT

### 6.1 104种语言训练一个BERT

谷歌的多语言BERT用104种语言训练做填空题。

**惊艳的发现**：
- 用**英文**问答数据微调多语言BERT
- 测试**中文**问答 → F1分数78.8%（与QANet的78.1%相当！）
- 从未见过中英互译数据，从未见过中文问答数据 → 自动学会了跨语言迁移！

### 6.2 为什么能跨语言？

对于多语言BERT，不同语言中同义的词，它们的嵌入向量非常接近：
- "兔子" ≈ "rabbit"
- "跳" ≈ "jump"
- "鱼" ≈ "fish"

但它并没有完全抹除语言信息——如果把中文句子的嵌入加上"英文-中文"差异向量，多语言BERT就会用中文来填空！

这甚至可以实现某种程度的**无监督词元级翻译**。

## 七、GPT：预测下一个词的惊人力量

### 7.1 GPT的预训练：文字接龙

- BERT = 双向编码器 + 填空（MLM）
- GPT = 单向解码器 + 预测下一个词

训练过程：
```
输入：<BOS>           → 预测："深"
输入：<BOS> 深        → 预测："度"
输入：<BOS> 深 度     → 预测："学"
输入：<BOS> 深 度 学  → 预测："习"
```

GPT使用Transformer的解码器架构，但采用**因果注意力掩码（Causal Mask）**——预测当前词时只能看到之前的词，不能看到之后的词。

### 7.2 In-Context Learning：不需要微调的学习

GPT-3（1750亿参数）太大，连微调都困难。但它有一种神奇的能力——**语境学习（In-Context Learning）**：

**零样本学习（Zero-shot）**：
```
把英文翻译成法文：cheese →
```

**单样本学习（One-shot）**：
```
把英文翻译成法文：sea → mer, cheese →
```

**小样本学习（Few-shot）**：
```
把英文翻译成法文：sea → mer, apple → pomme, dog → chien, cheese →
```

**关键**：整个过程**没有任何梯度下降**，没有更新任何参数！GPT只是根据上下文模式"推理"出答案。

> 这就是ChatGPT"听懂人话"的技术基础——规模足够大后，"预测下一个词"这个简单的任务能催生出理解、推理、翻译等"涌现能力"。

In [ ]:
# ============================================
# PyTorch示例2：GPT风格的因果注意力掩码
# ============================================
import torch
import torch.nn as nn

def create_causal_mask(seq_len):
    """
    创建因果注意力掩码（GPT的核心机制）
    
    确保位置i只能关注位置0到i，不能看到i+1及以后
    这保证了"预测下一个词"时不会"作弊"看到答案
    """
    # 创建一个上三角矩阵，上三角为1（被mask掉）
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    # 将被mask的位置设为负无穷（softmax后权重为0）
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

# 示例：seq_len=5
mask = create_causal_mask(5)
print("因果注意力掩码（GPT）：")
print("行=查询位置，列=可关注位置")
print("0=可关注，-inf=不可关注")
print()
for i in range(5):
    row_str = " ".join(["O" if mask[i,j] == 0 else "X" for j in range(5)])
    print(f"位置{i}: [{row_str}]")
print()
print("位置0只能看到自己；位置4能看到所有位置。")
print("这正是GPT的'只看左边'机制。")

## 八、BERT vs GPT：两大范式对比

| 特性 | BERT | GPT |
|------|------|------|
| **架构** | Transformer编码器 | Transformer解码器 |
| **注意力方向** | 双向（看整个上下文） | 单向/因果（只看左边） |
| **预训练任务** | MLM（填空） | 预测下一个词（文字接龙） |
| **强项** | 理解任务（分类、QA、NER） | 生成任务（写作、对话、翻译） |
| **微调** | 需要（加分类头训练） | GPT-3几乎不需要（In-Context） |
| **代表** | BERT, RoBERTa, ALBERT | GPT-3, GPT-4, ChatGPT |
| **预训练数据量** | 30亿词（BERT） | 45TB文本（GPT-3） |

> **直觉**：BERT像"阅读理解"专家（充分理解给定文本），GPT像"即兴创作"大师（根据上文续写下文）。两者互补。

## 九、自监督学习在其他领域的扩展

### 9.1 计算机视觉
- SimCLR：对比学习（同一图片的两个增强版本应产生相似的特征）
- BYOL：不需要负样本的对比学习
- MAE：类似BERT的掩码图像建模

### 9.2 语音
- 语音版BERT：掩码声音片段，预测被掩码的部分
- 语音版GPT：预测接下来会出现的声音
- SUPERB基准：语音领域的"GLUE"，10个不同任务全面评估
- s3prl工具包：包含各种自监督语音模型

### 9.3 Seq2Seq的预训练
BART、T5等模型预训练整个Seq2Seq架构：
- 编码器输入被损坏的句子
- 解码器输出还原的句子
- 损坏方式：掩码、删除、打乱顺序、旋转等
- T5尝试了所有可想象的组合（在C4数据集上训练，原始文件7TB）

## 十、跨章节连接

| 章节 | 连接关系 |
|------|----------|
| Ch6 自注意力 | BERT和GPT的核心机制都基于自注意力 |
| Ch7 Transformer | BERT是Transformer编码器，GPT是Transformer解码器 |
| Ch11 自编码器 | BERT可视为去噪文本自编码器（[MASK]=噪声） |
| Ch13 迁移学习 | 预训练→微调是迁移学习在NLP中的标准范式 |
| Ch15 元学习 | 好的初始化参数=预训练；元学习是另一个找好初始化的方向 |

## 十一、核心要点总结

1. **自监督学习**：从数据本身自动构造标签，无需人工标注。将数据分为输入和伪标签两部分。

2. **BERT**：双向Transformer编码器，预训练任务=MLM（填空）+NSP（下一句预测，后来发现用处不大）。

3. **BERT微调**：在预训练BERT上接任务特定的线性层，用少量标注数据微调。4种常见模式：单句分类、逐token分类、两句关系、抽取式问答。

4. **BERT为什么有效**：学会上下文相关词嵌入——"吃苹果"和"苹果手机"中的"果"有不同的向量表示。语言学理论（Firth, 1960）早已预言。

5. **多语言BERT**：104种语言训练，自动对齐不同语言的同义词嵌入。用英文问答训练，能零样本迁移到中文问答！

6. **GPT**：单向Transformer解码器，预训练=预测下一个词。规模足够大后涌现"In-Context Learning"能力——不需要梯度下降，给几个例子就能做翻译等任务。

7. **BERT vs GPT**：BERT适合理解（双向看全文），GPT适合生成（单向接龙）。两者互补。

8. **预训练→微调**：NLP的标准范式。预训练用海量无标注数据，微调用少量标注数据。训练BERT需要8天TPU或200天GPU。

9. **规模法则**：更大模型+更多数据=更强能力。从ELMo的94M到Switch Transformer的1.6T，参数量增长了17000倍。

10. **跨领域扩展**：自监督学习已扩展到计算机视觉（SimCLR、MAE）和语音（语音BERT/GPT），SUPERB基准全面评估语音SSL模型。

## 十二、练习与思考

1. 自监督学习与无监督学习有什么区别？为什么需要单独命名？

2. BERT的MLM任务中，为什么需要"随机替换"和"保持不变"两种策略，而不全部使用[MASK]？

3. 下一句预测（NSP）为什么用处不大？句序预测（SOP）为什么更好？

4. 为BERT设计一个用于"命名实体识别（NER）"的微调头。

5. 用Firth的语言学理论解释BERT为什么能学到上下文相关的词嵌入。

6. 多语言BERT是如何实现"零样本跨语言迁移"的？为什么用英文问答训练能回答中文问题？

7. GPT的In-Context Learning和传统微调有什么根本区别？为什么GPT-3能做In-Context Learning而BERT不能？

8. 实现GPT的因果注意力掩码，并在一个小的自回归语言模型上验证。

9. 为什么BERT做不了生成任务（如写文章），GPT做生成任务很自然？

10. 讨论：如果BERT的成功不完全来自"理解语义"（如DNA分类实验所示），那它成功的本质原因是什么？